# 01.04 Ollama — 로컬 LLM

`ChatOllama`는 로컬에서 돌아가는 Ollama daemon(기본 `http://localhost:11434`)에 HTTP로 붙는 클라이언트다. 개발·오프라인·프라이버시 민감 워크로드에 적합하다.

## 학습 목표

- Ollama daemon을 띄우고 `ChatOllama(model="llama3.2")`로 연결한다
- 로컬 모델 중 **tool calling 가능/불가** 호환성 표를 이해한다
- `num_ctx`, `num_gpu`, `reasoning`, `keep_alive` 등 로컬 튜닝 파라미터를 조정한다
- `base_url`로 원격 Ollama 서버(사내 GPU 박스 등) 에 붙인다

## 언제 쓰나

- 오프라인 / air-gapped 환경
- 민감 데이터(의료·법률·내부 문서)를 외부 API로 내보낼 수 없을 때
- GPU 박스 1대를 여러 서비스가 공유하는 사내 허브

## 01.04.1 환경 설정

사전 준비: (1) `brew install ollama` 등으로 설치, (2) `ollama serve` 로 daemon 기동, (3) `ollama pull llama3.2` 로 모델 다운로드.

probe는 `ChatOllama.invoke("ping")` — daemon 이 내려있거나 모델이 없으면 예외를 던진다.

In [ ]:
# !pip install -U langchain langchain-ollama
import os
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_ollama import ChatOllama
# daemon + 모델이 준비되었는지 확인 — 실패하면 이후 셀은 자동 스킵된다.
ChatOllama(model="llama3.2").invoke("ping")

## 01.04.2 기본 사용

In [ ]:
model = ChatOllama(model="llama3.2", temperature=0)
msg = model.invoke("로컬 Ollama의 가장 큰 장점을 한 문장으로 설명해줘.")
print(msg.content)

## 01.04.3 `init_chat_model()` 경로

In [ ]:
from langchain.chat_models import init_chat_model

m = init_chat_model("ollama:llama3.2")
print(m.invoke("핑").content[:80])

## 01.04.4 Tool calling 호환성

Ollama 모델 전부가 tool calling을 지원하지는 않는다. Ollama가 기능 비트를 모델 매니페스트에서 읽어 지원 모델에만 노출한다. **2026-04 현재** 지원 모델 일부:

| 모델 ID | tool calling | 비고 |
|---------|--------------|------|
| `llama3.2`, `llama3.1` | ✅ | Meta 공식 function-calling 학습 |
| `qwen2.5`, `qwen3` | ✅ | Alibaba Qwen 계열, 한국어 양호 |
| `mistral-small`, `mistral-large` | ✅ | Mistral 계열 |
| `firefunction-v2` | ✅ | Fireworks tuned |
| `llama2`, `mistral:7b` (구형) | ❌ | tool schema 미학습 |
| `phi3`, `gemma2` | ⚠️ 부분 | 일부 tag/버전만 |

아래 셀은 `llama3.2`에 도구를 bind 해 호출해 본다.

In [ ]:
from langchain_core.tools import tool

@tool
def mul(a: int, b: int) -> int:
    """두 정수를 곱한다."""
    return a * b

bound = model.bind_tools([mul])
out = bound.invoke("7 곱하기 6 은?")
print("tool_calls:", out.tool_calls)

## 01.04.5 로컬 튜닝 파라미터

로컬 추론 특유의 파라미터들:

In [ ]:
tuned = ChatOllama(
    model="llama3.2",
    num_ctx=8192,        # 컨텍스트 길이 (모델 최대치 내)
    num_gpu=99,          # GPU에 올릴 레이어 수 (99 = 가능한 최대)
    temperature=0.3,
    keep_alive="10m",    # daemon 메모리에서 모델을 10분 유지
)
print(tuned.invoke("한 문장으로 자기소개").content[:120])

## 01.04.6 원격 Ollama 서버 — `base_url`

사내 GPU 박스에 Ollama를 올려두고 개발 노트북에서 붙는 패턴.

In [ ]:
# 예시 (로컬이 아닌 경우):
# remote = ChatOllama(model="llama3.2", base_url="http://gpu-box.internal:11434")
# print(remote.invoke("원격 추론 테스트").content)
print("base_url 기본값:", getattr(model, 'base_url', 'http://localhost:11434'))

## 01.04.7 Structured output

In [ ]:
from pydantic import BaseModel

class City(BaseModel):
    name: str
    country: str

structured = model.with_structured_output(City)
print(structured.invoke("서울은 한국에 있어. 구조화해줘."))

## 체크리스트

- [ ] `ollama serve`로 daemon을 띄웠고 `ollama pull <model>`로 모델을 받았다
- [ ] 쓰려는 모델이 tool calling을 지원하는지 호환성 표에서 확인했다
- [ ] `num_ctx` / `num_gpu` / `keep_alive` 로 지연·메모리 트레이드오프를 맞췄다
- [ ] 원격 서버 공유 시 `base_url` 을 명시했다

## 다음

- `06_groq.ipynb` — 같은 오픈 가중치 모델을 관리형 고속 추론으로
- `05_bedrock.ipynb` — 엔터프라이즈 관리형 경로

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/chat/ollama
- Ollama 모델 허브: https://ollama.com/library